# Exploring DuckDB with iacs Data

This notebook loads iacs entity-centered data into DuckDB and lets you
explore it with SQL and the Python API.

In [ ]:
import duckdb
import pandas as pd

from iacs.io_system import IOSystem
from iacs.registry import Registry

## 1. Load iacs data into a Registry

In [ ]:
io = IOSystem()
entity_centered = io.read_entity_centered_file("../components/components.yaml")
registry = Registry.from_entity_centered(entity_centered)

print(f"Component types: {registry.component_types}")

## 2. Connect to DuckDB and register component tables

DuckDB can query pandas DataFrames directly — no import/export step needed.

In [ ]:
# In-memory database (nothing persisted to disk)
con = duckdb.connect()

# Register each component table as a DuckDB view
for comp_type in registry.component_types:
    df = registry.view(comp_type).reset_index()
    # DuckDB doesn't like spaces in table names, so replace them
    table_name = comp_type.replace(" ", "_")
    con.register(table_name, df)
    print(f"Registered '{table_name}' ({len(df)} rows)")

## 3. Explore with SQL

Now you can write SQL against your component tables.

In [ ]:
# List all registered tables
con.sql("SHOW TABLES").show()

In [ ]:
# Look at the description table
con.sql("SELECT * FROM description LIMIT 10").show()

In [ ]:
# Look at the id table — see entity paths and hashes
con.sql("SELECT entity_id, key, path, hash, alias FROM id LIMIT 10").show()

In [ ]:
# Join id and description to see human-readable entity info
con.sql("""
    SELECT id.path, id.key, description.value AS description
    FROM id
    JOIN description ON id.entity_id = description.entity_id
    LIMIT 20
""").show()

In [ ]:
# Count entities per component type
con.sql("""
    SELECT ct.component_type, COUNT(DISTINCT entity_id) AS n_entities
    FROM id
    CROSS JOIN (SELECT DISTINCT component_type FROM description) ct
    GROUP BY ct.component_type
""").show()

## 4. Explore with the Python relational API

DuckDB also has a relational API that feels like chaining DataFrame operations.

In [ ]:
# Get a relation object
desc = con.table("description")
desc.limit(5).show()

In [ ]:
# Filter and project
id_table = con.table("id")
id_table.filter("alias IS NOT NULL").project("entity_id, path, alias").show()

In [ ]:
# Join using the relational API
id_table.join(desc, "id.entity_id = description.entity_id").project(
    "id.path, description.value"
).limit(10).show()

In [ ]:
# Convert any result back to pandas
result_df = con.sql("SELECT * FROM requirement").df()
result_df.head()

## 5. Try your own queries

Use `con.sql("...")` for SQL or `con.table("...")` for the relational API.
Call `.show()` to display results or `.df()` to get a pandas DataFrame.